# Gold Test Evaluation — Cross-Model Comparison via DeepSeek API

Evaluates gold test predictions from 3 base models (Qwen3-0.6B, Qwen3-1.7B, Llama-3.2-1B) \
across multiple fine-tuning method variants.

**Metrics:** RAGAS (Faithfulness, Answer-Relevance, Context Precision, Context Recall)  
**Evaluation LLM:** DeepSeek-V4-Flash (OpenAI-compatible API)  
**Goal:** Compare which fine-tuning method performs best across model architectures.


## ⚡ Which files to evaluate?

Edit `SELECTED_CSVS` in the config cell (Section 3). It is a manual pick-list of
CSV stems from `NEW_BATCH_FILES` (the `new_batch` folder: 7 files across 3 models).

- `SELECTED_CSVS = ['qwen3_1_7b_scs_lora_no_prompt']` → evaluate only that file
- `SELECTED_CSVS = ['llama_3_2_1b_scs_lora', 'qwen3_0_6b_scs_lora']` → run those two, one CSV at a time
- `SELECTED_CSVS = []` → run nothing (safety valve)

**Run concurrently:** copy this notebook on Kaggle and give each copy a different
stem in `SELECTED_CSVS` — one CSV file per session, evaluated one file at a time.
Each copy writes to its own `/kaggle/working/<model>/<variant>/` folder.

**Resume:** already-evaluated variants are skipped via checkpointing (`eval_report.json` / `eval_per_sample.csv` already present).

**Thinking budget:** DeepSeek's API has no token-level thinking cap (only `thinking.type` and `reasoning_effort` low/high/max). The config pins `reasoning_effort = 'low'` for the cheapest chain-of-thought (default is `high`).


## 1. Dependencies

In [ ]:
%%capture
%pip install pandas matplotlib numpy tqdm
%pip install ragas langchain-openai 'langchain-community<0.4' python-dotenv sentence-transformers


## 2. Imports & Setup

In [ ]:
import os
import json
import random
import time
import threading
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from dotenv import load_dotenv
from openai import AsyncOpenAI

# -----------------------------------------------------------------------------
# Environment
# -----------------------------------------------------------------------------

warnings.filterwarnings("ignore")
load_dotenv()

# -----------------------------------------------------------------------------
# Device
# -----------------------------------------------------------------------------

EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {EMBED_DEVICE}")

# -----------------------------------------------------------------------------
# Ragas
# -----------------------------------------------------------------------------

from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)
from ragas.run_config import RunConfig
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings


class CompatHuggingFaceEmbeddings(HuggingFaceEmbeddings):
    """Bridge for ragas versions that dropped the LangChain-style
    embed_query / embed_documents interface while older metrics
    (e.g. ResponseRelevancy) still call them.
    """

    def embed_query(self, text: str):
        return self.embed_text(text)

    def embed_documents(self, texts):
        return self.embed_texts(texts)

    async def aembed_query(self, text: str):
        return await self.aembed_text(text)

    async def aembed_documents(self, texts):
        return await self.aembed_texts(texts)


print("Imports ready ✓")

## 3. Configuration

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Central config — edit paths / keys here
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_INPUT = Path(
    "/kaggle/input/datasets/tiamatt/thesis-improve-output/new_batch"
)
OUTPUT_ROOT = Path("/kaggle/working/")

MODEL_DIRS = {
    "qwen3-0.6b": BASE_INPUT / "qwen3-0.6b",
    "qwen3-1.7b": BASE_INPUT / "qwen3-1.7b",
    "llama-3.2-1b": BASE_INPUT / "llama-3.2-1b",
}

# --- Files in each model directory ---
# Keys = model directory
# Values = CSV stems without "_gold_test_predictions.csv"
NEW_BATCH_FILES = {
    "qwen3-0.6b": [
        "qwen3_0_6b_scs_lora",
    ],
    "qwen3-1.7b": [
        "qwen3_1_7b_question_type_mixture_lora_soft_routing",
        "qwen3_1_7b_scs_lora",
        "qwen3_1_7b_scs_lora_no_prompt",
        "qwen3_1_7b_scs_lora_prompt_scale_0_5_r16_a32",
        "qwen3_1_7b_scs_lora_prompt_scale_0_5_r4_a8",
        "qwen3_1_7b_scs_lora_scale_0_5",
        "qwen3_1_7b_scs_lora_scale_2",
    ],
    "llama-3.2-1b": [
        "llama_3_2_1b_scs_lora",
    ],
}

# --- Manual selection ---
# Evaluate one or more stems sequentially.
# [] = evaluate nothing.
SELECTED_CSVS = [
    # "qwen3_0_6b_scs_lora",
    # "qwen3_1_7b_question_type_mixture_lora_soft_routing",
    # "qwen3_1_7b_scs_lora",
    "qwen3_1_7b_scs_lora_no_prompt",
    # "qwen3_1_7b_scs_lora_prompt_scale_0_5_r16_a32",
    # "qwen3_1_7b_scs_lora_prompt_scale_0_5_r4_a8",
    # "qwen3_1_7b_scs_lora_scale_0_5",
    # "qwen3_1_7b_scs_lora_scale_2",
    # "llama_3_2_1b_scs_lora",
]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DeepSeek evaluator
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
DEEPSEEK_API_KEY = user_secrets.get_secret("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise ValueError(
        "DEEPSEEK_API_KEY was not found in Kaggle Secrets."
    )

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-v4-flash"

client = AsyncOpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

# Ragas uses the OpenAI-compatible client.
EVAL_LLM_PROVIDER = "openai"

# DeepSeek reasoning configuration.
DEEPSEEK_REASONING_EFFORT = "low"

llm = llm_factory(
    model=DEEPSEEK_MODEL,
    provider=EVAL_LLM_PROVIDER,
    client=client,
    max_tokens=16384,
    extra_body={
        "reasoning_effort": DEEPSEEK_REASONING_EFFORT,
    },
)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embeddings
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

embeddings = CompatHuggingFaceEmbeddings(
    model="BAAI/bge-m3",
    device=EMBED_DEVICE,
    normalize_embeddings=True,
    batch_size=32,
)

print(f"Embedding device: {EMBED_DEVICE}")
print("Embedding model : BAAI/bge-m3")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Ragas runtime
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RAGAS_TIMEOUT_PER_JOB = 420
RAGAS_MAX_WORKERS = 1
RAGAS_MAX_RETRIES = 3
RAGAS_BATCH_SIZE = 64
RAGAS_SAMPLE_FRAC = 1.0


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Heartbeat
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

HEARTBEAT_INTERVAL = 1800
HEARTBEAT_ENABLED = True


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Test mode
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

TEST_MODE = False
TEST_SAMPLE_N = 5


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Reproducibility
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("Config loaded ✓")
print(
    f"Eval LLM : DeepSeek | {DEEPSEEK_MODEL} "
    f"(reasoning_effort={DEEPSEEK_REASONING_EFFORT})"
)
print(
    f"Selected CSVs : {SELECTED_CSVS} "
    f"({len(SELECTED_CSVS)} file(s), one at a time)"
)
print(
    "Metrics : RAGAS "
    "(Faithfulness · Answer-Relevance · Context Precision · Context Recall)"
)
print(
    f"Heartbeat : every {HEARTBEAT_INTERVAL // 60} min "
    f"(enabled={HEARTBEAT_ENABLED})"
)

## 4. Discover Prediction CSV Files

In [ ]:
# Build model -> [csv paths] from the hardcoded NEW_BATCH_FILES stems (no glob discovery)
files_by_model = {}
missing = []
for model_name, d in MODEL_DIRS.items():
    wanted = sorted(NEW_BATCH_FILES.get(model_name, []))
    present = []
    for stem in wanted:
        p = d / f'{stem}_gold_test_predictions.csv'
        if p.exists():
            present.append(p)
        else:
            missing.append(f'{model_name}/{stem}')
    files_by_model[model_name] = present
    print(f'{model_name}: {len(present)}/{len(wanted)} files present')
    for f in present:
        print(f'  - {f.name}')

if missing:
    print(f'[WARN] missing on disk ({len(missing)}):')
    for m in missing:
        print(f'  - {m}')

total = sum(len(v) for v in files_by_model.values())
print(f'\nTotal prediction files: {total}')

## 5. Evaluation Functions

RAGAS metrics (Faithfulness · Answer-Relevance · Context Precision · Context Recall) + aggregate report builder


In [ ]:
def compute_ragas(
    df: pd.DataFrame,
    evaluator_llm,
    evaluator_embedding,
    batch_size: int = RAGAS_BATCH_SIZE,
    sample_fraction: float = 1.0,
    max_workers: int = 8,
    timeout: int = 300,
    seed: int = SEED
) -> pd.DataFrame:
    sampled_df = df.sample(frac=sample_fraction, random_state=seed).reset_index(drop=True)
    n = len(sampled_df)
    n_batches = (n + batch_size - 1) // batch_size
    print(f'Computing RAGAS metrics on {n} samples (batch size {batch_size}, {n_batches} batches)...', flush=True)
    metrics = [
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embedding, strictness=1),
        LLMContextPrecisionWithReference(llm=evaluator_llm),
        LLMContextRecall(llm=evaluator_llm),
    ]
    run_cfg = RunConfig(
        max_workers=max_workers,
        timeout=timeout,
        max_retries=RAGAS_MAX_RETRIES,
        max_wait=60,
        seed=SEED,
    )
    results = []
    hb = {'start': time.time(), 'batch_start': time.time(), 'done': 0}
    hb_stop = threading.Event()

    def _heartbeat():
        while not hb_stop.wait(HEARTBEAT_INTERVAL):
            elapsed = time.time() - hb['start']
            cur = time.time() - hb['batch_start']
            print(f'[HEARTBEAT] RAGAS progress: {hb["done"]}/{n_batches} batches done '
                  f'| elapsed {elapsed/60:.1f} min | current batch running {cur/60:.1f} min',
                  flush=True)

    hb_thread = threading.Thread(target=_heartbeat, daemon=True)
    if HEARTBEAT_ENABLED:
        hb_thread.start()

    try:
        for i in range(n_batches):
            hb['batch_start'] = time.time()
            batch_df = sampled_df.iloc[i*batch_size:(i+1)*batch_size]
            samples = [
                SingleTurnSample(
                    user_input=row["question"],
                    response=row["prediction"],
                    retrieved_contexts=(
                        row["context_used"] if isinstance(row["context_used"], list)
                        else [row["context_used"]]
                    ),
                    reference=row["answer"],
                )
                for _, row in batch_df.iterrows()
            ]
            eval_dataset = EvaluationDataset(samples=samples)
            result = evaluate(
                dataset=eval_dataset,
                metrics=metrics,
                llm=evaluator_llm,
                run_config=run_cfg,
            )
            results.append(result.to_pandas())
            hb['done'] = i + 1
            print(f'  batch {i+1}/{n_batches} done — {hb["done"] * batch_size}/{n} samples', flush=True)
    finally:
        if HEARTBEAT_ENABLED:
            hb_stop.set()
            hb_thread.join(timeout=2)
    return pd.concat(results, ignore_index=True)


METRIC_LABELS = [
    'Faithfulness', 'Answer-Relevance', 'Context Relevance', 'Context Recall',
]

_COL_TO_LABEL = {
    'faithfulness':                             'Faithfulness',
    'answer_relevancy':                         'Answer-Relevance',
    'llm_context_precision_with_reference':     'Context Relevance',
    'context_recall':                           'Context Recall',
}


def _build_report(df: pd.DataFrame) -> dict:
    def _agg(subset: pd.DataFrame) -> dict:
        row = {'n_samples': len(subset)}
        for col, label in _COL_TO_LABEL.items():
            if col in subset.columns:
                val = subset[col].mean()
                row[label] = round(float(val), 4) if pd.notna(val) else float('nan')
            else:
                row[label] = float('nan')
        return row

    report = {}
    for qt, grp in df.groupby('question_type', sort=True):
        report[str(qt).upper()] = _agg(grp)
    report['OVERALL'] = _agg(df)
    return report


def evaluate_all(
    df_pred: pd.DataFrame,
    evaluator_llm,
    evaluator_embedding,
    ragas_sample_frac: float = 1.0,
    seed: int = SEED,
) -> tuple:
    df = df_pred.copy().reset_index(drop=True)

    # Map gold_question_type -> question_type
    if 'gold_question_type' in df.columns and 'question_type' not in df.columns:
        df['question_type'] = df['gold_question_type']

    ragas_cols = [
        "faithfulness",
        "answer_relevancy",
        "llm_context_precision_with_reference",
        "context_recall"
    ]
    for col in ragas_cols:
        df[col] = float("nan")

    n_ragas = max(1, int(len(df) * ragas_sample_frac))
    ragas_idx = df.sample(n=n_ragas, random_state=seed).index
    df_ragas_input = df.loc[ragas_idx].reset_index(drop=False)
    original_indices = df_ragas_input["index"].tolist()
    df_ragas_input = df_ragas_input.drop(columns=["index"])

    print(f'[RAGAS] Computing RAGAS metrics on {n_ragas}/{len(df)} samples '
          f'({ragas_sample_frac:.0%}) in batches of {RAGAS_BATCH_SIZE}...')

    ragas_scores = compute_ragas(
        df=df_ragas_input,
        evaluator_llm=evaluator_llm,
        evaluator_embedding=evaluator_embedding,
        batch_size=RAGAS_BATCH_SIZE,
        sample_fraction=1.0,
        max_workers=RAGAS_MAX_WORKERS,
        timeout=RAGAS_TIMEOUT_PER_JOB,
        seed=seed,
    )

    for col in ragas_cols:
        if col in ragas_scores.columns:
            df.loc[original_indices, col] = ragas_scores[col].values

    report = _build_report(df)
    print(f"Done. QA types in report: {[k for k in report if k != 'OVERALL']}")
    return df, report


print('Evaluation functions ready \u2713')

## 6. Run Evaluation (Checkpointed)

Each variant is evaluated independently. Results are saved immediately \
so interrupted runs can resume without recomputing completed variants.

In [ ]:
all_reports = {}  # {model_name: {variant_name: report_dict}}


# --- Heartbeat (disabled by default; set HEARTBEAT_ENABLED=True in the config cell to enable):
#     one CSV file is evaluated at a time; report progress every HEARTBEAT_INTERVAL ---
def _start_file_heartbeat(label: str):
    """Print a progress heartbeat while the current CSV file is being evaluated."""
    evt = threading.Event()
    t0 = time.time()

    def _loop():
        while not evt.wait(HEARTBEAT_INTERVAL):
            print(f'[HEARTBEAT] still evaluating {label} '
                  f'| file elapsed {(time.time() - t0) / 60:.1f} min', flush=True)

    th = threading.Thread(target=_loop, daemon=True)
    if HEARTBEAT_ENABLED:
        th.start()
    return evt, th


# Resolve the manually selected CSVs (by stem) -> (model, path) pairs
selected = [
    (m, p)
    for m, lst in files_by_model.items()
    for p in lst
    if p.stem.replace('_gold_test_predictions', '') in SELECTED_CSVS
]
missing_sel = [
    s for s in SELECTED_CSVS
    if s not in {p.stem.replace('_gold_test_predictions', '') for lst in files_by_model.values() for p in lst}
]
if missing_sel:
    print(f'[WARN] selected stem(s) not found on disk: {missing_sel}')
print(f'Selected {len(selected)} file(s) to evaluate (one CSV at a time):')
for m, p in selected:
    print(f'  {m}: {p.name}')
print()

if not selected:
    found = [p.name for lst in files_by_model.values() for p in lst]
    print(f'[ERROR] SELECTED_CSVS={SELECTED_CSVS} matched no files on disk.')
    print(f'        Files actually found ({len(found)}): {found}')
    raise SystemExit(
        'No CSV files matched SELECTED_CSVS. Check that the dataset is attached '
        'at BASE_INPUT and SELECTED_CSVS matches a stem in NEW_BATCH_FILES.'
    )

for model_name, csv_path in selected:
    variant = csv_path.stem.replace('_gold_test_predictions', '')
    out_dir = OUTPUT_ROOT / model_name / variant
    out_dir.mkdir(parents=True, exist_ok=True)

    report_path = out_dir / 'eval_report.json'
    per_sample_path = out_dir / 'eval_per_sample.csv'

    if report_path.exists() and per_sample_path.exists():
        try:
            with open(report_path, encoding='utf-8') as f:
                report = json.load(f)
        except (json.JSONDecodeError, OSError):
            report = None
        if report is not None:
            print(f'[SKIP] {model_name}/{variant} — already evaluated')
            all_reports.setdefault(model_name, {})[variant] = report
            continue
        print(f'[WARN] {report_path} exists but is invalid — re-evaluating')

    print(f'\n{"="*60}')
    print(f'Evaluating: {model_name} / {variant}   (one CSV file at a time)')
    print(f'{"="*60}')

    hb_evt, hb_th = _start_file_heartbeat(f'{model_name}/{variant}')
    try:
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
        print(f'Samples: {len(df)}')
        if TEST_MODE:
            df = df.sample(TEST_SAMPLE_N, random_state=SEED).reset_index(drop=True)
            print(f'[TEST MODE] sampled {TEST_SAMPLE_N} rows for a quick pipeline test')

        result_df, report = evaluate_all(
            df, llm, embeddings,
            ragas_sample_frac=RAGAS_SAMPLE_FRAC,
        )

        if not TEST_MODE:
            result_df.to_csv(per_sample_path, index=False, encoding='utf-8-sig')
            with open(report_path, 'w', encoding='utf-8') as f:
                json.dump(report, f, ensure_ascii=False, indent=2)

        all_reports.setdefault(model_name, {})[variant] = report
        print(f'[SAVED] {out_dir}' if not TEST_MODE
              else f'[TEST] {model_name}/{variant} done (results not saved to checkpoint)')
    finally:
        if HEARTBEAT_ENABLED:
            hb_evt.set()
            hb_th.join(timeout=2)

print(f'\n{"="*60}')
print('Evaluation complete —')
total_variants = sum(len(v) for v in all_reports.values())
print(f'Total variants evaluated: {total_variants}')

## 7. Per-Model Results Summary

In [ ]:
DISPLAY_METRICS = ['Faithfulness', 'Answer-Relevance', 'Context Relevance', 'Context Recall']

def print_model_report(model_name: str, reports: dict):
    print(f'\n{"═"*60}')
    print(f'  {model_name}')
    print(f'{"═"*60}')
    for variant, report in sorted(reports.items()):
        stats = report.get('OVERALL', {})
        print(f'\n  [{variant}]  n={stats.get("n_samples", "?")}')
        for m in DISPLAY_METRICS:
            v = stats.get(m, float('nan'))
            if pd.isna(v):
                print(f'    {m:<22} n/a')
                continue
            bar = '█' * int(v * 25) + '░' * (25 - int(v * 25))
            print(f'    {m:<22} {bar}  {v:.4f}')

for model_name, reports in all_reports.items():
    print_model_report(model_name, reports)

## 8. Cross-Method Comparison

Group variants by fine-tuning method (extracted from filename) and compare \
performance across all 3 models. This reveals which method generalizes best.

In [ ]:
def extract_method(variant_name: str) -> str:
    """Method name = the variant stem (new_batch names already encode the method)."""
    return variant_name

# Build comparison table
KEY_METRICS = ['Faithfulness', 'Answer-Relevance', 'Context Relevance', 'Context Recall']

rows = []
for model_name, reports in all_reports.items():
    for variant, report in reports.items():
        method = extract_method(variant)
        overall = report.get('OVERALL', {})
        n = overall.get('n_samples', 0)
        row = {'model': model_name, 'method': method, 'n_samples': n}
        for m in KEY_METRICS:
            row[m] = overall.get(m, float('nan'))
        rows.append(row)

comp_df = pd.DataFrame(rows, columns=['model', 'method', 'n_samples'] + KEY_METRICS)

if len(comp_df) == 0:
    print('[WARN] No evaluated variants in this session — nothing to compare.')
    print('       Run Section 6 first, or check that the dataset is attached and')
    print('       SELECTED_CSVS matches a stem in NEW_BATCH_FILES.')
    pivot = pd.DataFrame()
    method_means = comp_df.groupby('method')[KEY_METRICS].mean()
    best_df = pd.DataFrame(columns=['metric', 'best_method', 'mean_score'])
else:
    print('=== Cross-Method Comparison Table (OVERALL means) ===')
    print(comp_df.round(4).to_string(index=False))
    print()

    # Pivot: methods as rows, metrics as columns with (model, value) pairs
    pivot = comp_df.pivot_table(
        index='method',
        columns='model',
        values=KEY_METRICS,
        aggfunc='first'
    )
    print('=== Pivot Table (methods × models) ===')
    print(pivot.round(4).to_string())
    print()

    # Compute cross-model mean per method per metric
    method_means = comp_df.groupby('method')[KEY_METRICS].mean()
    print('=== Method Means (averaged across models) ===')
    print(method_means.round(4).to_string())
    print()

    # Best method per metric
    best_methods = []
    for m in KEY_METRICS:
        s = method_means[m].dropna()
        if len(s) == 0:
            best_methods.append({'metric': m, 'best_method': 'n/a', 'mean_score': float('nan')})
        else:
            best_methods.append({'metric': m, 'best_method': s.idxmax(), 'mean_score': round(float(s.max()), 4)})
    best_df = pd.DataFrame(best_methods)
    print('=== Best Method per Metric (by cross-model mean) ===')
    print(best_df.to_string(index=False))

## Notes

- **RAGAS** is evaluated on `RAGAS_SAMPLE_FRAC` (1.0 = all samples) per variant to manage API cost.
- **Checkpointing** means you can interrupt and resume; completed variants are skipped.
- The `.env` file at `{BASE_DIR}/.env` must contain `DEEPSEEK_API_KEY=sk-...`.
